In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

# Standard library - import pathlib with explicit alias to avoid matplotlib.path conflict
import sys
import time
import re
import random
from pathlib import Path as PathLib

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import sklearn
from sklearn.cluster import KMeans, DBSCAN
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
import umap
import anndata as ad
import scanpy as sc
from sknetwork.clustering import Louvain, Leiden
from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
import xgboost as xgb
import distinctipy
import networkx
from leidenalg import find_partition
import shap
import icecream as ic

# Add custom module paths


# Jupyter/IPython magic commands
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Plotting configuration
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

params = {
    'axes.titlesize': 30,
    'legend.fontsize': 20,
    'figure.figsize': (6, 5),
    'axes.labelsize': 20,
    'axes.titlesize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'figure.titlesize': 30
}
plt.rcParams.update(params)
sns.set_style("white")

# Analysis configuration
Run = "Corrs"
hKWD = {'element': 'step', 'fill': False, 'stat': 'density'}
pKWD = {'dpi': 200, 'bbox_inches': 'tight'}

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
# CyTOF Helper Package
import cytof_helper
from cytof_helper.utils import get_markers, gate_cells
from cytof_helper.normalization import normalize_markers_optimization
from cytof_helper.plotting import *
from cytof_helper.stats import *


# Histogram Plotting
Using `plot_histograms_multi_df` from `cytof_helper.plotting`.

# Load and initialize

In [ ]:
# Define data directory using pathlib
data_dir = PathLib("/Users/ronguy/Dropbox/IDH_CyTOF/DKFZ_project_IDH/20251113_32D_timeLaps1/output")

In [ ]:
# Use pathlib glob to get file list
FList = list(data_dir.glob("*"))

In [ ]:
# Sort files by numerical identifier
FList = sorted(FList, key=lambda s: int(re.search(r"[cC](\d+)", s.name).group(1)))

In [ ]:
DBs=[f"c{x:02}" for x in range(1,17)]

In [ ]:
DBs

In [ ]:
R={'c01':'WT',
 'c02':'WT_8h',
 'c03':'WT_24h',
 'c04':'WT_72h',
 'c05':'Mut',
 'c06':'Mut_8h',
 'c07':'Mut_24h',
 'c08':'Mut_72h',
 'c09':'WT_Ac',
 'c10':'WT_Ac_8h',
 'c11':'WT_Ac_24h',
 'c12':'WT_AC_72h',
 'c13':'Mut_Ac',
 'c14':'Mut_Ac_8h',
 'c15':'Mut_Ac_24h',
 'c16':'Mut_Ac_72h'}

In [ ]:
DBs=[R[DB] for DB in DBs]

In [ ]:
for F,DB in zip(FList,DBs):
    print(F)
    globals()[DB]=pd.read_csv(F)

In [ ]:
# Load mapping file using pathlib
mapping_file = PathLib("Mapping.xlsx")
Rep=dict(pd.read_excel(mapping_file).iloc[:,:].values)

In [ ]:
Rep

In [ ]:
for F,DB in zip(FList,DBs):
    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)


In [ ]:
N=list(WT.columns)
N.sort()

In [ ]:
N

In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=get_markers(N,marker_file='Markers_Names.xlsx')

In [ ]:
plot_histograms_multi_df([np.arcsinh(globals()[db]/5) for db in DBs],
                         N,df_names=DBs,hist_kwargs=hKWD,ncols=4);

In [ ]:
len(NamesAll)

In [ ]:
for DB in DBs:
    plt.figure()
    D=np.arcsinh(globals()[DB]/5).copy()
    sns.histplot(data=D,x='H3',**hKWD,color='blue')
    sns.histplot(data=D,x='H3.3',**hKWD,color='red')
    sns.histplot(data=D,x='H4',**hKWD,color='magenta')
#    sns.histplot(data=D,x='H2A',**hKWD,color='g')
    plt.title(DB)
#plt.xscale('log')
#plt.yscale('log')

## Gate on H3.3/H2A too low, but also remove outliers 99.99% from all

# Gating
Using `gate_cells` from `cytof_helper.utils`.

In [ ]:
for DB in DBs:
    globals()[DB]=gate_cells(globals()[DB], name=DB)

In [ ]:
plot_histograms_multi_df([globals()[db] for db in DBs],
                         N,df_names=DBs,hist_kwargs=hKWD,ncols=4);

# Normalize using new method on all intercellular markers

# Normalization
Using `normalize_markers_optimization` from `cytof_helper.normalization` package instead of defining it inline.

In [ ]:
NormMRK

In [ ]:
for DB in DBs:
    globals()[DB]=normalize_markers_optimization(globals()[DB],norm_columns=['H3','H3.3','H4'],norm_markers=NormMRK)

In [ ]:
scFac=5
for DB in DBs:
    globals()[DB]=np.arcsinh(globals()[DB]/scFac)

In [ ]:
MRK_All=NamesAll.copy()
MRK_All.remove('H3')
MRK_All.remove('H3.3')
MRK_All.remove('H4')
#MRK_All.remove('H2A')

EPC=EpiCols.copy()
Core=['H3','H3.3','H4']#,'H2A']
for C in Core:
    EPC.remove(C)

In [ ]:
NC=2000
aaaa=pd.concat([globals()[DB].sample(NC, replace=False, random_state=RANDOM_SEED) for DB in DBs]).copy()
                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
    globals()[DB]['Line']=DB

In [ ]:
DBCLR=dict(zip(DBs,distinctipy.get_colors(len(DBs))))

In [ ]:
plot_histograms_multi_df([globals()[db] for db in DBs],
                         MRK_All+['mCD11b'],df_names=DBs,hist_kwargs=hKWD,ncols=4);

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

# Your original aggregation
s = (
    pd.concat([globals()[db] for db in DBs])
      .groupby('Line')['mCD11b']
      .mean()
)

df = s.rename('mCD11b').reset_index()

# Extract hour from suffix; no suffix -> 0h
df['hour'] = df['Line'].str.extract(r'_(\d+)h$')[0].fillna(0).astype(int)

# Remove time suffix to get sample name
df['sample'] = df['Line'].str.replace(r'_(\d+)h$', '', regex=True)

# Normalize naming inconsistencies (AC vs Ac)
df['sample'] = df['sample'].str.replace(r'_AC$', '_Ac', regex=True)

# Keep only lines with expected timepoints, then sort
df = df[df['hour'].isin([0, 8, 24, 72])].sort_values(['sample', 'hour'])

# Plot one line per sample with real x-intervals
fig, ax = plt.subplots(figsize=(8, 5))
for sample, g in df.groupby('sample'):
    ax.plot(g['hour'], g['mCD11b'], marker='o', linewidth=2, label=sample)

ax.set_xticks([0, 8, 24, 72])
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Mean mCD11b')
ax.set_title('mCD11b over time by sample')
ax.legend(title='Sample', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('Plots/mCD11b_Time.png',dpi=200,bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Full data
df = pd.concat([globals()[db] for db in DBs], ignore_index=True)[['Line', 'mCD11b']].copy()

# Parse time (0h if no suffix) and sample name
df['hour'] = df['Line'].str.extract(r'_(\d+)h$')[0].fillna(0).astype(int)
df['sample'] = df['Line'].str.replace(r'_(\d+)h$', '', regex=True)

# Normalize naming inconsistency
df['sample'] = df['sample'].str.replace(r'_AC$', '_Ac', regex=True)

# Keep expected timepoints
time_order = [0, 8, 24, 72]
df = df[df['hour'].isin(time_order)]

# Optional: stable sample order
sample_order = sorted(df['sample'].unique())

# Violin plots: one panel per timepoint, all samples in each panel
g = sns.catplot(
    data=df,
    x='sample', y='mCD11b',
    col='hour', col_order=time_order,
    kind='violin', order=sample_order,
    cut=0, inner='quartile', sharey=True,
    height=4, aspect=1.1
)

g.set_axis_labels("Sample", "mCD11b")
g.set_titles("{col_name}h")
for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
NamesAll

In [ ]:
MRK_All=[
 
 'H3K27ac',
 'H3K27me3',
 'H3K4me1',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me3',
 
 'H4K16ac',
 'KI67',
 
# 'mCD11b',
# 'mLy-6G',
 'pRb']

In [ ]:
CAll=pd.concat([globals()[DB] for DB in DBs]).copy()


In [ ]:
Mat=CAll.groupby('Line').mean()

In [ ]:
Mat=Mat.loc[DBs,:]

In [ ]:
plt.figure(figsize=(5,10))
sns.clustermap(np.round(Mat[MRK_All].T,2),annot=True,cmap=plt.cm.seismic,center=0,yticklabels=True,col_cluster=True)
plt.xticks(fontsize=12);
plt.yticks(fontsize=12);
plt.savefig(f'Plots/IDH_All_2.png',**pKWD)

In [ ]:
plt.figure(figsize=(5,10))
sns.clustermap(np.round(Mat[MRK_All].T,2),annot=True,cmap=plt.cm.seismic,center=0,yticklabels=True,col_cluster=False)
plt.xticks(fontsize=12);
plt.yticks(fontsize=12);
plt.savefig(f'Plots/IDH_All.png',**pKWD)

In [ ]:
WDBs=['WT',
 'WT_8h',
 'WT_24h',
 'WT_72h',
 'Mut',
 'Mut_8h',
 'Mut_24h',
 'Mut_72h',
]

In [ ]:
NC=2000
CAll=pd.concat([globals()[DB].sample(NC, replace=False, random_state=RANDOM_SEED) for DB in WDBs]).copy()


In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=42,verbose=True)

X_2d=UM.fit_transform(CAll[MRK_All])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
AD=ad.AnnData(CAll[NamesAll],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
MRK=NamesAll.copy()
MRK=list(set(MRK).difference(set(['H3','H3.3','H4','Line'])))

In [ ]:
MRK.sort()
AD=ad.AnnData(CAll[MRK],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
sc.pp.neighbors(AD, random_state=RANDOM_SEED)

In [ ]:
sc.tl.leiden(AD, resolution=0.5, flavor="igraph", n_iterations=2, random_state=RANDOM_SEED)

In [ ]:
sc.pl.umap(AD,color=MRK+['Line','leiden'],cmap='seismic',vmin='p01',vmax='p99',hspace=0.5,wspace=0.5,ncols=3,show=False)
plt.savefig("Plots/UMAPs.png",dpi=200,bbox_inches='tight')

In [ ]:
DF=AD.to_df()
DF['lbl']=AD.obs['leiden'].astype(int)
DF['Line']=AD.obs['Line']

In [ ]:
import pandas as pd

cluster_col = 'lbl'   # change to your cluster column name
df = DF.copy()

# Parse genotype/time and a "condition" that pairs WT vs Mut correctly (e.g., '' vs '_Ac')
df['time_h'] = df['Line'].str.extract(r'_(\d+)h$')[0].fillna(0).astype(int)
df['genotype'] = df['Line'].str.extract(r'^(WT|Mut)')
df['condition'] = (
    df['Line']
      .str.replace(r'^(WT|Mut)', '', regex=True)   # remove genotype prefix
      .str.replace(r'_(\d+)h$', '', regex=True)    # remove time suffix
)

# Keep only WT/Mut
d = df[df['genotype'].isin(['WT', 'Mut'])].copy()

# % cells in each cluster within each (genotype, condition, time)
cnt = (
    d.groupby(['genotype', 'condition', 'time_h', cluster_col], observed=True)
     .size()
     .rename('n')
     .reset_index()
)
tot = (
    d.groupby(['genotype', 'condition', 'time_h'], observed=True)
     .size()
     .rename('N')
     .reset_index()
)

pct = cnt.merge(tot, on=['genotype', 'condition', 'time_h'])
pct['pct_cells'] = 100 * pct['n'] / pct['N']

# Mean of WT and Mut percentages: (WT + Mut)/2
mean_wt_mut = (
    pct.groupby(['condition', 'time_h', cluster_col], observed=True)['pct_cells']
       .mean()
       .rename('mean_pct_WT_Mut')
       .reset_index()
)

# Optional: also keep WT and Mut side-by-side
wide = (
    pct.pivot_table(
        index=['condition', 'time_h', cluster_col],
        columns='genotype',
        values='pct_cells',
        aggfunc='mean'
    )
    .reset_index()
)
wide['mean_pct_WT_Mut'] = wide[['WT', 'Mut']].mean(axis=1)

# main result:
mean_wt_mut


In [ ]:
for i in mean_wt_mut['lbl'].unique():
    print(f"Cluster {i}")
    print(mean_wt_mut[mean_wt_mut['lbl']==i])

In [ ]:
Mat=DF.groupby('lbl').mean(numeric_only=True).iloc[[4,0,2,
                                                    6,5,
                                                    1,
                                                    7,3,8],:]
sns.clustermap(Mat.T,cmap='seismic',center=0,vmax=2,annot=True,col_cluster=False)
plt.savefig("Plots/HM1.png",dpi=200,bbox_inches='tight')

In [ ]:
M=DF.Line.str.contains('WT')
Mat=DF.groupby('lbl').mean(numeric_only=True).iloc[[4,0,2,
                                                    6,5,
                                                    1,
                                                    7,3,8],:]
sns.clustermap(Mat.T,cmap='seismic',center=0,vmax=2,annot=True,col_cluster=False)
plt.savefig("Plots/HM1_WT.png",dpi=200,bbox_inches='tight')

In [ ]:
M=DF.Line.str.contains('Mut')
Mat=DF.groupby('lbl').mean(numeric_only=True).iloc[[4,0,2,
                                                    6,5,
                                                    1,
                                                    7,3,8],:]
sns.clustermap(Mat.T,cmap='seismic',center=0,vmax=2,annot=True,col_cluster=False)
plt.savefig("Plots/HM1_Mut.png",dpi=200,bbox_inches='tight')

In [ ]:
cg = sns.clustermap(
    Mat.T,
    cmap='seismic',
    center=0,
    vmax=2,
    annot=True,
    col_cluster=False
)

ax = cg.ax_heatmap
n_cols = Mat.T.shape[1]
if n_cols < 6:
    raise ValueError(f"Need at least 6 columns, got {n_cols}")

# first 3 -> 0h, next 2 -> 8h, next 1 -> 24h, rest -> 72h
time_labels = ['0h'] * 3 + ['8h'] * 2 + ['24h'] + ['72h'] * (n_cols - 6)

# keep existing labels, add time in a row below via newline
orig = [t.get_text() for t in ax.get_xticklabels()]
ax.set_xticklabels([f"{o}\n{t}" for o, t in zip(orig, time_labels)], rotation=0, ha='center');

cg.fig.savefig("Plots/HM2.png", dpi=200, bbox_inches='tight')


In [ ]:
DF[DF['lbl']==3].Line.value_counts()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

order_lbl = [4, 0, 2, 6, 5, 1, 7, 3, 8]
geno_order = ['WT', 'Mut']  # WT then Mut for each cluster

DF2 = DF.copy()
DF2['Genotype'] = DF2['Line'].str.extract(r'^(WT|Mut)', expand=False)
DF2 = DF2.dropna(subset=['Genotype'])

# mean per (cluster, genotype)
Mat2 = (
    DF2.groupby(['lbl', 'Genotype'])
       .mean(numeric_only=True)
       .reindex(pd.MultiIndex.from_product([order_lbl, geno_order], names=['lbl', 'Genotype']))
)

# optional: drop missing cluster/genotype combos if any
Mat2 = Mat2.dropna(how='all')

col_labels = [f"{lbl}_{geno}" for lbl, geno in Mat2.index]

cg = sns.clustermap(
    Mat2.T,
    cmap='seismic',
    center=0,
    vmax=2,
    annot=True,
    col_cluster=False,
    xticklabels=col_labels
)

cg.ax_heatmap.tick_params(axis='x', rotation=90)
cg.fig.savefig("Plots/HM3.png", dpi=200, bbox_inches='tight')


In [ ]:
CM=pd.crosstab(DF['Line'],DF['lbl'])

In [ ]:
sns.heatmap(CM/CM.sum(0),annot=True,cmap='magma')

In [ ]:
LBLS=['WT','WT_8h','WT_24h','WT_72h','Mut','Mut_8h','Mut_24h','Mut_72h']

In [ ]:
CM = pd.crosstab(DF['lbl'], DF['Line'])[LBLS]
CM_normalized = CM / CM.sum(0)

# Stacked barplot - transpose so columns become bars
ax = CM_normalized.T.plot(kind='bar', stacked=True, cmap='tab10', figsize=(10, 6),
                          edgecolor='white', linewidth=0.5)

# Add annotations on each segment
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='center', fontsize=8)

plt.ylabel('Proportion')
plt.xlabel('Line')
plt.xticks(rotation=90)
plt.legend(title='lbl', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('Plots/Lines.png',dpi=200,bbox_inches='tight')
plt.show()

In [ ]:
DF.groupby('lbl').size()

In [ ]:
CM_normalized

In [ ]:
CM = pd.crosstab(DF['Line'], DF['lbl'])
CM_normalized = CM / CM.sum(0)
CM_normalized=CM_normalized.iloc[:,[4,0,2,
                                                    6,5,
                                                    1,
                                                    7,3,8]]
# Stacked barplot - transpose so columns become bars
ax = CM_normalized.T.plot(kind='bar', stacked=True, cmap='tab10', figsize=(10, 6),
                          edgecolor='white', linewidth=0.5)

# Add annotations on each segment
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='center', fontsize=8)

plt.ylabel('Proportion')
plt.xlabel('lbl')
plt.xticks(rotation=0)
plt.legend(title='Line', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('Plots/Clusters.png',dpi=200,bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DF2 = DF.copy()

# Build WT/Mut label safely
DF2['Genotype'] = pd.NA
DF2.loc[DF2['Line'].str.startswith('WT', na=False), 'Genotype'] = 'WT'
DF2.loc[DF2['Line'].str.startswith('Mut', na=False), 'Genotype'] = 'Mut'
DF2 = DF2.dropna(subset=['Genotype'])

CM = pd.crosstab(DF2['Genotype'], DF2['lbl'])
CM_normalized = CM.div(CM.sum(axis=0), axis=1)
CM_normalized=CM_normalized.iloc[:,[4,0,2,
                                                    6,5,
                                                    1,
                                                    7,3,8]]
ax = CM_normalized.T.plot(
    kind='bar', stacked=True, cmap='tab10', figsize=(10, 6),
    edgecolor='white', linewidth=0.5
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='center', fontsize=8)

plt.ylabel('Proportion')
plt.xlabel('lbl')
plt.xticks(rotation=0)
plt.legend(title='Genotype', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('Plots/Clusters2.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
fig,ax=plt.subplots(4,4,figsize=(20,20))
a=ax.flatten()
for i,L in enumerate(CAll['Line'].unique()):
    M=CAll['Line']==L
    a[i].scatter(X_2d[M,0],X_2d[M,1],s=1,label=L)
    a[i].legend()
    
#plt.legend()

In [ ]:
WDBs=[
    
 'WT_8h',
 'Mut_8h',
 'WT_Ac_8h',
 'Mut_Ac_8h',
]

In [ ]:
NC=4000
CAll=pd.concat([globals()[DB].sample(NC, replace=False, random_state=RANDOM_SEED) for DB in WDBs]).copy()


In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=42,verbose=True)

X_2d=UM.fit_transform(CAll[MRK_All])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
AD=ad.AnnData(CAll[NamesAll],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
MRK=NamesAll.copy()
MRK=list(set(MRK).difference(set(['H3','H3.3','H4','Line'])))

In [ ]:
MRK.sort()
AD=ad.AnnData(CAll[MRK],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
sc.pp.neighbors(AD, random_state=RANDOM_SEED)

In [ ]:
sc.tl.leiden(AD, resolution=0.5, flavor="igraph", n_iterations=2, random_state=RANDOM_SEED)

In [ ]:
sc.pl.umap(AD,color=MRK+['Line','leiden'],cmap='seismic',vmin='p01',vmax='p99',hspace=0.5,wspace=0.5,ncols=3,show=False)
plt.savefig("Plots/UMAPs_8h.png",dpi=200,bbox_inches='tight')

In [ ]:
fig,ax=plt.subplots(4,4,figsize=(20,20))
a=ax.flatten()
for i,L in enumerate(CAll['Line'].unique()):
    M=CAll['Line']==L
    a[i].scatter(X_2d[M,0],X_2d[M,1],s=1,label=L)
    a[i].legend()
    
#plt.legend()

In [ ]:
DF=AD.to_df()
DF['lbl']=AD.obs['leiden'].astype(int)
DF['Line']=AD.obs['Line']

In [ ]:
LBLS=WDBs

In [ ]:
CM = pd.crosstab(DF['lbl'], DF['Line'])[LBLS]
CM_normalized = CM / CM.sum(0)

# Stacked barplot - transpose so columns become bars
ax = CM_normalized.T.plot(kind='bar', stacked=True, cmap='tab10', figsize=(10, 6),
                          edgecolor='white', linewidth=0.5)

# Add annotations on each segment
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='center', fontsize=8)

plt.ylabel('Proportion')
plt.xlabel('Line')
plt.xticks(rotation=90)
plt.legend(title='lbl', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('Plots/Lines_8h.png',dpi=200,bbox_inches='tight')
plt.show()

In [ ]:
DF.groupby('lbl').size()

In [ ]:
CM_normalized

In [ ]:
CM = pd.crosstab(DF['Line'], DF['lbl'])
CM_normalized = CM / CM.sum(0)
# CM_normalized=CM_normalized.iloc[:,[4,0,2,
#                                                     6,5,
#                                                     1,
#                                                     7,3,8]]
# # Stacked barplot - transpose so columns become bars
ax = CM_normalized.T.plot(kind='bar', stacked=True, cmap='tab10', figsize=(10, 6),
                          edgecolor='white', linewidth=0.5)

# Add annotations on each segment
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='center', fontsize=8)

plt.ylabel('Proportion')
plt.xlabel('lbl')
plt.xticks(rotation=0)
plt.legend(title='Line', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('Plots/Clusters_8h.png',dpi=200,bbox_inches='tight')

plt.show()

In [ ]:
Mat=DF.groupby('lbl').mean(numeric_only=True)
sns.clustermap(Mat.T,cmap='seismic',center=0,vmax=2,annot=True,col_cluster=True)
plt.savefig("Plots/HM1_8h.png",dpi=200,bbox_inches='tight')